# Upgraded Pipeline 3 — R4 relation validation (GPU) & pre/post benchmark

Consumes `benchmarks/r4_bundle_2026-07-25.zip` (built locally, CPU): the 3 real AG/RIVE
sheets' GIVEN GPT-5.5 entities run through the deterministic relationship pipeline in BOTH
configs (original / upgraded = backbone pass + line-label filter), one crop per unique
candidate edge in the v3-relation adapter's trained format.

This notebook runs **only the GPU half**: Qwen3-VL-8B + v3-relation adapter validates each
candidate (keep/reject), producing the **post-LLM** relation sets. Combined with the
**pre-LLM** sets already in the bundle, that gives the full 2x2:

| | pre-LLM (deterministic) | post-LLM (R4 validated) |
|---|---|---|
| **original Pipeline 3** | set A | set B |
| **upgraded Pipeline 3** | set C | set D |

No relation ground truth exists for these real sheets, so the 4 sets + their crops are
pushed back to HF for **Opus high-effort adjudication** (the GT-substitute). Both the
pre-LLM result (stands alone if R4 hurts) and the post-LLM result are preserved.

**Honest expectation (Probe 2, 2026-07-24):** the local Qwen relation-validator scored
chance-level on real crops, so R4 may REJECT real edges and degrade the sets. That is a
result worth measuring in-pipeline, not assuming — and the pre-LLM sets are the fallback.

In [1]:
!nvidia-smi

Sat Jul 25 07:05:14 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   33C    P0             56W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 1. Config

In [2]:
import os
# Token is read from the environment, never hardcoded. In Colab add it under
# Secrets (key: HF_TOKEN) and enable notebook access; locally just export it.
HF_TOKEN = os.environ.get("HF_TOKEN", "")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception as e:
        raise RuntimeError("HF_TOKEN is not set - add it to Colab Secrets or export it") from e
DATA_REPO = "timthy45/pnid-extraction-datasets"
CKPT_REPO = "timthy45/qwen3vl-pnid-domain-base"
QWEN_MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct"
RELATION_ADAPTER_PATH = "v3-relation/latest"   # the local relation-validator candidate

BUNDLE_FILE = "benchmarks/r4_bundle_2026-07-25.zip"
RESULTS_PATH_IN_REPO = "benchmarks/r4_validation_results_2026-07-25.json"

# Smoke first: cap candidates per sheet, flip to None for the full run once the gate passes.
MAX_CANDIDATES_PER_SHEET = 8   # -> None for full

assert HF_TOKEN.startswith("hf_")

## 2. Install (GPU-runtime prep)

`torchao` is uninstalled first: peft's LoRA dispatcher probes it and errors on the pinned
version even though no quantization is used here (same fix as the other v3 notebooks).

In [3]:
!pip uninstall -y torchao -q
!pip install -q "transformers==4.57.1" peft huggingface_hub pillow

import torch
assert torch.cuda.is_available(), "No GPU - set Runtime type first"
print(torch.cuda.get_device_name(0))

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 144.3 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 50.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.
NVIDIA A100-SXM4-80GB


## 3. Fetch the pre-built candidate bundle

In [4]:
import json, zipfile
from pathlib import Path
from huggingface_hub import hf_hub_download

BUNDLE_ROOT = Path("/content/r4_bundle")
_zp = hf_hub_download(repo_id=DATA_REPO, filename=BUNDLE_FILE, repo_type="dataset",
                      token=HF_TOKEN, force_download=True)
BUNDLE_ROOT.mkdir(exist_ok=True)
with zipfile.ZipFile(_zp) as zf:
    zf.extractall(BUNDLE_ROOT)

manifest = json.load(open(BUNDLE_ROOT / "manifest.json"))
n_candidates = sum(len(s["candidates"]) for s in manifest["sheets"].values())
print(f"sheets: {list(manifest['sheets'])}")
for sid, s in manifest["sheets"].items():
    print(f"  {sid}: original={len(s['original_pairs'])} upgraded={len(s['upgraded_pairs'])} "
          f"unique-candidates={len(s['candidates'])}")
print(f"total unique candidate crops: {n_candidates}")

benchmarks/r4_bundle_2026-07-25.zip:   0%|          | 0.00/43.1M [00:00<?, ?B/s]

sheets: ['PX-2368-0180004-001', 'GD-B-540-DP-2920-005-Z', 'PX-2365-0140006-001']
  PX-2368-0180004-001: original=78 upgraded=52 unique-candidates=83
  GD-B-540-DP-2920-005-Z: original=140 upgraded=201 unique-candidates=240
  PX-2365-0140006-001: original=106 upgraded=150 unique-candidates=169
total unique candidate crops: 492


## 4. Load Qwen3-VL-8B + attach the v3-relation adapter

In [5]:
from transformers import AutoModelForImageTextToText, AutoProcessor
from peft import PeftModel
from huggingface_hub import snapshot_download

processor = AutoProcessor.from_pretrained(QWEN_MODEL_ID)
base_model = AutoModelForImageTextToText.from_pretrained(
    QWEN_MODEL_ID, dtype=torch.bfloat16, device_map="cuda").eval()
print("Qwen base loaded. VRAM:", f"{torch.cuda.memory_allocated()/1e9:.1f} GB")

_local = Path("/content/v3_relation_adapter")
snapshot_download(repo_id=CKPT_REPO, repo_type="model", token=HF_TOKEN,
                  allow_patterns=[f"{RELATION_ADAPTER_PATH}/*"], local_dir=str(_local))
_dir = _local / RELATION_ADAPTER_PATH
assert (_dir / "adapter_model.safetensors").exists(), f"missing: {_dir}"
model = PeftModel.from_pretrained(base_model, str(_dir), adapter_name="relation")
model.set_adapter("relation")
print("v3-relation adapter attached and active")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

adapter_config.json: 0.00B [00:00, ?B/s]

progress.json:   0%|          | 0.00/27.0 [00:00<?, ?B/s]

v3-relation/latest/adapter_model.safeten(…):   0%|          | 0.00/1.43G [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:1348: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(msg)


v3-relation adapter attached and active


In [6]:
import base64, io
from PIL import Image

def qwen_generate(image, prompt, max_tokens=16):
    messages = [{"role": "user", "content": [
        {"type": "image", "image": image}, {"type": "text", "text": prompt}]}]
    inputs = processor.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_dict=True, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_tokens, do_sample=False)
    t = out[0][inputs["input_ids"].shape[1]:]
    return processor.decode(t, skip_special_tokens=True).strip()

## 5. R4 relation validation — one bounded keep/reject per candidate

Prompt matches the v3-relation adapter's exact trained format (bracket pixel coords, forced
yes/no) from `build_relation_pool` — its fairest shot. `yes` -> keep the candidate edge,
`no` or unparseable -> reject.

In [7]:
import re

def r4_prompt(a_local, b_local):
    return (f"In this P&ID crop there is a symbol at [{a_local}] and another at "
            f"[{b_local}] (pixel coordinates, top-left origin). Are these two symbols "
            f"directly connected to each other? Answer yes or no.")

def parse_keep(raw):
    t = raw.strip().lower()
    if t.startswith("yes"):
        return True
    if t.startswith("no"):
        return False
    return None   # unparseable -> treated as reject below

verdicts = {}   # (sheet_id, "a|b") -> {"keep": bool, "raw": str}
for sid, s in manifest["sheets"].items():
    cands = s["candidates"]
    if MAX_CANDIDATES_PER_SHEET:
        cands = cands[:MAX_CANDIDATES_PER_SHEET]
    for c in cands:
        img = Image.open(BUNDLE_ROOT / c["crop_file"]).convert("RGB")
        raw = qwen_generate(img, r4_prompt(c["a_local"], c["b_local"]))
        keep = parse_keep(raw)
        key = (sid, f"{c['pair'][0]}|{c['pair'][1]}")
        verdicts[key] = {"keep": bool(keep), "raw": raw, "pair": c["pair"],
                         "in_original": c["in_original"], "in_upgraded": c["in_upgraded"]}
    n_yes = sum(1 for k, v in verdicts.items() if k[0] == sid and v["keep"])
    n_tot = sum(1 for k in verdicts if k[0] == sid)
    print(f"{sid}: R4 kept {n_yes}/{n_tot} candidates ({n_yes/n_tot:.0%} yes)" if n_tot else sid)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


PX-2368-0180004-001: R4 kept 0/8 candidates (0% yes)
GD-B-540-DP-2920-005-Z: R4 kept 0/8 candidates (0% yes)
PX-2365-0140006-001: R4 kept 0/8 candidates (0% yes)


## 6. Assemble the 2x2 and report

Counts only — no relation GT for these real sheets, so quality is Opus's call (next cell
pushes the sets + crops for adjudication). A big drop from pre to post = R4 rejecting a lot;
whether those were real edges is exactly what Opus decides.

In [8]:
rows = []
for sid, s in manifest["sheets"].items():
    orig_pre = {tuple(p) for p in s["original_pairs"]}
    upg_pre = {tuple(p) for p in s["upgraded_pairs"]}
    # post = pre intersected with R4-kept (only candidates we actually validated)
    kept = {tuple(v["pair"]) for k, v in verdicts.items() if k[0] == sid and v["keep"]}
    validated = {tuple(v["pair"]) for k, v in verdicts.items() if k[0] == sid}
    # a pre-edge survives to post iff it was validated AND kept; un-validated pre-edges
    # (only when MAX_CANDIDATES caps the run) are excluded from post to keep it honest
    orig_post = {p for p in orig_pre if p in kept}
    upg_post = {p for p in upg_pre if p in kept}
    rows.append((sid, len(orig_pre), len(orig_post), len(upg_pre), len(upg_post),
                 len(validated)))

print(f"{'sheet':28} {'origPre':>8} {'origPost':>9} {'upgPre':>7} {'upgPost':>8} {'validated':>10}")
for r in rows:
    print(f"{r[0]:28} {r[1]:>8} {r[2]:>9} {r[3]:>7} {r[4]:>8} {r[5]:>10}")
if MAX_CANDIDATES_PER_SHEET:
    print(f"\n[SMOKE RUN: only {MAX_CANDIDATES_PER_SHEET} candidates/sheet validated. "
          "Set MAX_CANDIDATES_PER_SHEET=None and re-run cells 5-7 for the full result.]")

sheet                         origPre  origPost  upgPre  upgPost  validated
PX-2368-0180004-001                78         0      52        0          8
GD-B-540-DP-2920-005-Z            140         0     201        0          8
PX-2365-0140006-001               106         0     150        0          8

[SMOKE RUN: only 8 candidates/sheet validated. Set MAX_CANDIDATES_PER_SHEET=None and re-run cells 5-7 for the full result.]


## 7. Save + push results for Opus adjudication

In [9]:
from huggingface_hub import HfApi

results = {
    "generated": "notebook run (fill date via args)",
    "smoke_cap": MAX_CANDIDATES_PER_SHEET,
    "sheets": {},
}
for sid, s in manifest["sheets"].items():
    kept = {tuple(v["pair"]) for k, v in verdicts.items() if k[0] == sid and v["keep"]}
    orig_pre = [tuple(p) for p in s["original_pairs"]]
    upg_pre = [tuple(p) for p in s["upgraded_pairs"]]
    results["sheets"][sid] = {
        "original_pre":  [list(p) for p in orig_pre],
        "original_post": [list(p) for p in orig_pre if tuple(p) in kept],
        "upgraded_pre":  [list(p) for p in upg_pre],
        "upgraded_post": [list(p) for p in upg_pre if tuple(p) in kept],
        "r4_verdicts": {f"{v['pair'][0]}|{v['pair'][1]}": {"keep": v["keep"], "raw": v["raw"]}
                        for k, v in verdicts.items() if k[0] == sid},
        "entities": s["entities"],
    }

out_path = "/content/r4_validation_results.json"
with open(out_path, "w") as f:
    json.dump(results, f, indent=1)
HfApi(token=HF_TOKEN).upload_file(
    path_or_fileobj=out_path, path_in_repo=RESULTS_PATH_IN_REPO,
    repo_id=DATA_REPO, repo_type="dataset", token=HF_TOKEN)
print(f"pushed -> {RESULTS_PATH_IN_REPO}")
print("Next: Opus high-effort pulls this + the bundle crops, adjudicates each relation "
      "in all 4 sets as real/not-real -> precision per set.")

pushed -> benchmarks/r4_validation_results_2026-07-25.json
Next: Opus high-effort pulls this + the bundle crops, adjudicates each relation in all 4 sets as real/not-real -> precision per set.


In [10]:
# free the GPU when done
import torch, gc
del model, base_model
gc.collect(); torch.cuda.empty_cache()
print("GPU freed. Unassign the runtime.")

GPU freed. Unassign the runtime.
